# ML Masterclass 01: Math & Data Foundations

Welcome to the first notebook of the **Machine Learning (0 to Masterclass) Sequence**. 

This track is designed strictly for AI researchers and senior engineers. We will not teach you how to call `import sklearn`. We will teach you the mathematical bedrock beneath `sklearn`, how to implement that math from scratch, and how it behaves when deployed to production.

---

## 🎓 Socratic Deep Check
Before we write any code, ponder this question. If you don't know the answer now, you will by the end of this notebook:

> *"If Maximum Likelihood Estimation (MLE) is simply finding the peak of a probability distribution, why do ML frameworks ALWAYS minimize the Negative Log-Likelihood instead of maximizing the raw Likelihood?"*

---

## Table of Contents
1. **Linear Algebra Reality Check:** Beyond matrices, into vector spaces and Eigendecomposition.
2. **Calculus & Optimization:** Implementing an Automatic Differentiation engine from scratch.
3. **Probability & Statistics:** MLE, MAP, and the mathematical justification for Loss Functions.
4. **Data Reality:** Imputation strategies and why standardizing data changes the physics of optimization.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors memorize formulas; seniors understand the GEOMETRY behind them. Eigendecomposition isn't abstract math — it's PCA, which compresses 10,000-pixel images into 50 numbers. Every ML algorithm is just linear algebra + calculus + probability working together.

**Mental Model:** Think of the covariance matrix as a magnifying glass that reveals which features move together. Eigenvectors point in the direction of maximum variance (the 'interesting' directions), and eigenvalues tell you how much information each direction carries.

**Common Junior Pitfall:** Applying PCA without standardizing features first. If Feature A ranges [0, 1000] and Feature B ranges [0, 1], PCA will think Feature A is infinitely more important simply because its scale is bigger — even if Feature B is the true predictor.

---


## 1. Linear Algebra Reality Check

In standard courses, you learn that a matrix is an Excel spreadsheet. In ML, a matrix is a **Linear Transformation**. 
When you multiply a vector $\vec{x}$ by a weight matrix $W$, you are physically rotating and stretching that vector into a new geometric space.

### The Curse and Cure of Dimensionality: Eigendecomposition

Why do we care about Eigenvectors? 
An eigenvector is a vector whose *direction* does not change when a specific linear transformation (matrix $A$) is applied to it. It is only stretched (scaled) by its corresponding Eigenvalue $\lambda$.

$A\vec{v} = \lambda\vec{v}$

In ML, the eigenvectors of a data covariance matrix represent the **Principal Components** (the axes of maximum variance). This is how PCA (Dimensionality Reduction) compresses a 10,000-pixel image into 50 numbers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Deriving Principal Components manually
# -----------------------------------------------------

# 1. Create a highly correlated 2D dataset (e.g., height vs weight)
np.random.seed(42)
X = np.random.multivariate_normal(mean=[0, 0], cov=[[5, 4], [4, 5]], size=500)

# 2. Center the data (crucial for covariance math)
X_centered = X - np.mean(X, axis=0)

# 3. Calculate the Covariance Matrix
# Cov(X, Y) = E[(X - E[X])(Y - E[Y])]
cov_matrix = np.cov(X_centered, rowvar=False)
print("Covariance Matrix:\n", cov_matrix)

# 4. Calculate Eigenvalues and Eigenvectors of the Covariance Matrix
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Sort them in descending order of eigenvalue (importance)
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]

print("\nEigenvalues (Variance explained):", eigenvalues)
print("Eigenvectors (Direction of variance):\n", eigenvectors)

### 🎓 DEEP QUESTION ANSWERED
# Q: Why does standardizing data (mean=0, variance=1) drastically change Gradient Descent?
# A: Look at the Eigenvalues of your dataset's Hessian/Covariance. 
# If feature A ranges from 0-1 and feature B ranges from 0-1000, the gradients for B 
# will explode, while A barely moves. The loss landscape is a steep, squished ravine. 
# Standardizing makes the loss landscape spherical, so gradient steps aim directly at the minimum.

## 2. Calculus & Optimization

Deep Learning frameworks (PyTorch, TensorFlow) feel like magic because you build a model, call `.backward()`, and the gradients calculate themselves. This is **Automatic Differentiation (AutoDiff)**.

AutoDiff is NOT numeric differentiation $\frac{f(x+h) - f(x)}{h}$ (too slow/inaccurate). 
AutoDiff is NOT symbolic differentiation (too complex for loops and branches).
AutoDiff is the programmatic application of the **Chain Rule**: $\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$ tracking operations dynamically on a computational graph.


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Micrograd-style AutoDiff from Scratch
# -----------------------------------------------------

class Value:
    """A single scalar value that mathematically tracks its own gradient."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0          # The derivative of the final output wrt to this value
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        # The local derivative of addition is 1.0 (it just passes the gradient through)
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        # By product rule, the local derivative of (x * y) wrt x is y.
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
        
    def backward(self):
        # Topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        # Go one variable at a time and apply the chain rule
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# Let's test it: f(x, y) = x * y + 2
# We want to find df/dx and df/dy at x=3, y=-4

x = Value(3.0, label='x')
y = Value(-4.0, label='y')
const = Value(2.0, label='const')

xy = x * y; xy.label = 'xy'
f = xy + const; f.label = 'f'

f.backward() # This single call triggers the chain rule backward through the whole expression

print(f"Function Evaluation f(3, -4): {f.data}")
print(f"df/dx (should be y, which is -4.0): {x.grad}")
print(f"df/dy (should be x, which is 3.0): {y.grad}")

# This concept is exactly what is happening inside PyTorch tensors!

## 3. Probability & Statistics (MLE and Loss Functions)

Why do we use **Mean Squared Error (MSE)** for regression and **Cross-Entropy** for classification?

It isn't arbitrary. Loss functions are derived directly from the **Maximum Likelihood Estimate (MLE)** under specific probability distributions.

*   **Assumption 1:** If we assume our target variable $y$ contains target signals *plus* Gaussian (Normally distributed) random noise, the mathematical MLE derivation simplifies exactly to minimizing **Mean Squared Error**.
*   **Assumption 2:** If we assume our target variable $y$ is binary (Bernoulli distribution), the MLE derivation simplifies exactly to minimizing **Log Loss (Cross-Entropy)**.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why do we ALWAYS minimize the Negative Log-Likelihood instead of maximizing the raw Likelihood?*

**A:** Two purely mathematical/computational reasons:
1. **Underflow:** Likelihood is the product of many probabilities ($P_1 * P_2 * ... * P_n$). Multiplying 10,000 numbers that are all between 0 and 1 results in a number so infinitesimally small that standard computer floats truncate it to exactly `0.0`. Taking the log turns multiplication into addition ($\log(A*B) = \log(A) + \log(B)$), preventing float underflow safely.
2. **Optimization conventions:** Gradient Descent algorithms are written to *minimize* "Loss" landscapes. Maximizing Likelihood is algebraically identical to minimizing the *negative* of the Log-Likelihood.

## 4. Data Reality: Missing Values & Imputation

In Kaggle, data is dense. In production, 30% of your values are missing. How you fill them dictates your model's bias.

1. **Mean/Median Imputation:** Fast, but destroys feature variance and correlations. Only use if data is missing completely at random (MCAR).
2. **Predictive (KNN/Iterative) Imputation:** Uses other features to guess the missing value. Powerful, but incredibly computationally expensive to run in real-time production.
3. **Missingness as a Feature:** *Why* is it missing? Often, the fact that a user didn't fill out 'Income' predicts credit default better than their actual income.


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: KNN Imputation from Scratch
# -----------------------------------------------------
from sklearn.metrics.pairwise import nan_euclidean_distances

def custom_knn_imputer(X, k=2):
    """Predicts missing values by finding the K most similar rows."""
    X_imputed = X.copy()
    
    # nan_euclidean ignores nan features when calculating distance 
    distances = nan_euclidean_distances(X, X)
    
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            if np.isnan(X[i, j]):
                # Get distances to all other points from point i
                dists = distances[i]
                
                # Sort indices by distance (excluding itself if distance is 0)
                sorted_idx = np.argsort(dists)
                valid_neighbors = []
                
                # Find k nearest neighbors that actually HAVE a value for feature j
                for idx in sorted_idx:
                    if idx != i and not np.isnan(X[idx, j]):
                        valid_neighbors.append(X[idx, j])
                    if len(valid_neighbors) == k:
                        break
                
                # Impute with the mean of the nearest neighbors
                if valid_neighbors:
                    X_imputed[i, j] = np.mean(valid_neighbors)
                else:
                    # Fallback to column mean if no neighbors exist
                    X_imputed[i, j] = np.nanmean(X[:, j])
                    
    return X_imputed

# Test dataset with missing values (NaN)
dataset = np.array([
    [1.0, 2.0, np.nan], # Missing feature 3
    [1.1, 2.1, 0.5],
    [9.0, 8.0, 5.0],
    [9.1, 8.2, 5.2],
    [1.2, 1.9, 0.6]
])

print("Original Dataset with missing values:\n", dataset)

# Impute missing feature 3. 
# It should look at Row 1, realize Row 2 and Row 5 are the closest mathematically, 
# and average their feature 3 values (0.5 and 0.6) resulting in ~0.55.
imputed_data = custom_knn_imputer(dataset, k=2)

print("\nImputed Dataset:\n", imputed_data)


---
## ✅ Knowledge Check

### Q1: Why Negative Log-Likelihood?

<details><summary>👉 Answer</summary>

Three reasons: (1) Products of tiny probabilities underflow to 0.0 on computers — log converts products to sums, which are numerically stable. (2) Minimizing negative log-likelihood is mathematically identical to maximizing likelihood, and all optimizers are built to MINIMIZE. (3) The resulting loss surface is convex for exponential family distributions, guaranteeing a global minimum.
</details>

### Q2: Eigenvalue interpretation

<details><summary>👉 Answer</summary>

The eigenvalue tells you what FRACTION of total variance that principal component explains. If λ₁ = 8.5 and λ₂ = 1.5 (total = 10), then PC1 captures 85% of the information. You can discard PC2 and lose only 15% of the data's structure — massive compression with minimal loss.
</details>

### Q3: Standardization before PCA

<details><summary>👉 Answer</summary>

You MUST standardize (zero mean, unit variance) before PCA. Without it, PCA finds the direction of maximum SCALE, not maximum INFORMATION. A feature in dollars [0, 1M] will dominate over a feature in percentages [0, 1], regardless of which is more predictive. StandardScaler ensures all features compete on equal footing.
</details>


---
## 🔨 Practical Practice

### Exercise 1: Implement PCA from scratch using only NumPy (no sklearn). Steps: center data, compute covariance matrix, find eigenvalues/eigenvectors, project data onto top-k components. Verify your result matches `sklearn.decomposition.PCA`.

### Exercise 2: Build a numerical gradient checker: compute ∂L/∂w using the limit definition (f(w+ε) - f(w-ε)) / 2ε and compare to your analytical gradient. This is how PyTorch's `gradcheck()` works internally.

### Exercise 3: Given a dataset with 50 features, use the explained variance ratio to determine the optimal number of PCA components that capture 95% of the variance. Plot the cumulative explained variance curve.
